In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Habilita renderizacao grafica inline no Jupyter notebook
%matplotlib inline

# Pre-processar EEG e criar janelas

**Dificuldade 1-2** | **Tempo de execucao: 2m** | **Computacao: CPU**

Sinais brutos de EEG raramente estao prontos para treinamento de modelos: taxa de amostragem incorreta, derivas lentas e ruido de rede eletrica, ausencia de referencia padronizada, transientes esporadicos de grande amplitude e uma linha do tempo continua em vez de epocas estruturadas. Este tutorial apresenta a receita canonica de pre-processamento do EEGDash em uma gravacao do [OpenNeuro](https://openneuro.org) ``ds002718`` (Wakeman & Henson 2015), acessivel atraves do [NEMAR](https://nemar.org) (Delorme et al. 2022). Cada decisao metodologica e nomeada explicitamente (Cisotto & Chicco 2024 Dicas 4-5), inspecionada nos arrays e finalizada com um conjunto de dados janelado que os quatro tutoriais seguintes utilizam. A figura diagnostica final compara o tracado antes e depois da rotina :class:`braindecode.preprocessing.EEGPrep`, que encapsula ASR (Mullen et al. 2015), deteccao de canais ruins, filtro passa-alta e referencia media comum.

.. sphinx_gallery_thumbnail_path = '_static/thumbs/plot_10_preprocess_and_window.png'
Palavras-chave: pre-processamento, janelamento, ASR


## Objetivos de aprendizagem
- Descrever o pipeline de pre-processamento como uma sequencia ordenada de transformacoes de arrays com proposito unico por etapa.
- Identificar metodos expostos por :class:`mne.io.Raw` e rotinas em :mod:`mne.preprocessing`.
- Definir montagem, referencia, filtragem e reamostragem usando parametros nomeados com :class:`braindecode.preprocessing.Preprocessor`.
- Converter dados continuos em janelas de comprimento fixo com formato ``(n_channels, window_samples)`` usando :func:`braindecode.preprocessing.create_fixed_length_windows`.
- Aplicar :class:`braindecode.preprocessing.EEGPrep` (pipeline baseado em ASR, Mullen et al. 2015) e inspecionar o que foi alterado em um painel diagnostico de 4 graficos.



## Requisitos
- Cerca de 3 min em CPU na primeira execucao; menos de 60 s quando em cache.
- Rede na primeira chamada (~80 MB gravados em ``cache_dir``); offline posteriormente.
- Pre-requisito: ``plot_01_first_recording``.
- Conceito: :doc:`/concepts/preprocessing_decisions`.



Configuracao inicial. O pre-processamento e deterministico dados os parametros estipulados, portanto sem semente (*seed*).



In [ ]:
# Importa modulos para graficos, MNE para processamento eletrofisiologico e tabelas pandas
import matplotlib.pyplot as plt
import mne
import pandas as pd

# Importa o modulo eegdash e rotinas de pre-processamento do Braindecode
import eegdash
from braindecode.preprocessing import (
    EEGPrep,
    Preprocessor,
    create_fixed_length_windows,
    preprocess,
)
from eegdash import EEGDashDataset
from eegdash.paths import get_default_cache_dir
from eegdash.viz import style_figure, use_eegdash_style

# Configura backend do MNE e nivel de registros de log
mne.viz.set_browser_backend("matplotlib")
mne.set_log_level("WARNING")
use_eegdash_style()

# Define parametros globais do pipeline de pre-processamento
CACHE_DIR = get_default_cache_dir()
TARGET_SFREQ = 200.0  # Hz, compativel com a taxa de calibracao do ASR
WINDOW_SIZE_S = 2.0  # Duracao da janela em segundos
L_FREQ, H_FREQ = 1.0, 40.0  # Limites do filtro passa-banda em Hz
EEGPREP_SLICE_S = 30.0  # Janela de 30s para o painel diagnostico
print(f"eegdash {eegdash.__version__}; cache_dir={CACHE_DIR}")

## Conceitos por tras do pre-processamento
Tres principios metodologicos devem ser observados:

1. **A ordem das operacoes nao e comutativa.** Uma montagem atribui posicoes 3D aos canais; a referencia media e calculada sobre esses eletrodos, portanto a montagem deve estar definida primeiro. A filtragem altera amplitudes canal por canal, de modo que pode comutar com a referencia, mas deve anteceder a reamostragem (evitando distorcoes nas bordas pelo novo limite de Nyquist). O janelamento vem ao final; uma vez cortado o sinal, perde-se a linha do tempo continua.
2. **Decisoes documentadas e nomeadas.** Cisotto & Chicco (2024) Dica 4 preconizam relatar o tipo de filtro, banda de passagem, fase e metodo de projeto. A Dica 5 aborda a escolha de referencia. Cada :class:`~braindecode.preprocessing.Preprocessor` declara esses parametros explicitamente, permitindo a reproducao fiel do fluxo.
3. **Dois niveis de aplicacao.** :class:`mne.io.Raw` aplica modificacoes in-place em uma gravacao isolada; :func:`~braindecode.preprocessing.preprocess` aplica a lista de operacoes em todas as gravacoes de um :class:`~eegdash.api.EEGDashDataset` mantendo os metadados vinculados.



## Evidencias sobre o que ajuda e o que prejudica

Kessler et al. (2025), *Communications Biology*, avaliaram diferentes escolhas de filtragem, referencia, linha de base, remocao de tendencia e quatro metodos de correcao de artefatos em experimentos do ERP CORE (40 participantes) com EEGNet e regressao logistica. Cada etapa de correcao de artefatos reduziu o desempenho de decodificacao em ambos os modelos; cortes passa-alta mais elevados elevaram a acuracia de forma consistente. Os autores alertam que artefatos nao corrigidos podem inflar a acuracia as custas da interpretabilidade: o modelo pode aprender padroes de ruido em vez de sinal neural genuino.

Delorme (2023), *Scientific Reports*, comparou pipelines otimizados do EEGLAB, FieldTrip, MNE e Brainstorm. Apenas uma configuracao superou a simples filtragem passa-alta. Etapas agressivas de referencia e correcao de linha de base foram prejudiciais; rejeitar segmentos ruins nao recuperou o poder estatistico perdido; a rejeicao automatica por ICA de componentes oculares e musculares frequentemente falhou em agregar ganho mensuravel.

Conclusao pratica: mantenha a receita sucinta. Ajuste a frequencia de corte do passa-alta antes de empilhar correcoes automatizadas. Empregue ICA ou ASR (Mullen et al. 2015; Kothe & Makeig 2013) somente apos mensurar que trazem beneficio real na sua tarefa especifica.



## O que um objeto Raw pode fazer?
Liste os metodos expostos por :class:`mne.io.Raw` para compreender as operacoes disponiveis nativamente.



In [ ]:
# Mapeia e exibe os metodos publicos da classe BaseRaw do MNE
raw_methods = sorted(
    name
    for name in dir(mne.io.BaseRaw)
    if not name.startswith("_") and callable(getattr(mne.io.BaseRaw, name, None))
)
pd.DataFrame({"method": raw_methods}).head(25)

## O que ha em :mod:`mne.preprocessing`?
Alem dos metodos do proprio objeto ``Raw``, o modulo :mod:`mne.preprocessing` disponibiliza rotinas avancadas: :class:`~mne.preprocessing.ICA`, :class:`~mne.preprocessing.EOGRegression`, projetores espaciais e detectores de artefatos.



In [ ]:
# Lista ferramentas disponiveis no submodulo mne.preprocessing
prep_attrs = sorted(
    name for name in dir(mne.preprocessing) if not name.startswith("_")
)[:25]
pd.DataFrame({"mne.preprocessing": prep_attrs})

## Etapa 1: Carregar uma gravacao (carregamento preguicoso / lazy)
Construa o dataset, selecione a gravacao e acesse ``record.raw`` para materializar o arquivo em memoria.



In [ ]:
# Define parametros BIDS da gravacao alvo
DATASET = "ds002718"
SUBJECT = "002"
TASK = "FaceRecognition"
# Inicializa o dataset pelo EEGDash
dataset = EEGDashDataset(
    cache_dir=CACHE_DIR, dataset=DATASET, subject=SUBJECT, task=TASK
)
# Carrega a gravacao completa em memoria RAM e gera uma copia de trabalho
record = dataset.datasets[0]
raw = record.raw.load_data().copy()
raw

**Preveja.** Qual e o formato do array obtido por ``raw.get_data()``? Canais por amostras, onde amostras = ``sfreq * duration``.



In [ ]:
# Extrai o array de dados do sinal e documenta suas propriedades
data_in = raw.get_data()
pd.DataFrame(
    {
        "value": [
            f"{data_in.shape}",
            str(data_in.dtype),
            f"{raw.info['sfreq']:.1f}",
            f"{raw.info['nchan']}",
            f"{raw.times[-1]:.1f}",
        ]
    },
    index=["raw.get_data().shape", "dtype", "sfreq (Hz)", "n_channels", "duration (s)"],
)

## Etapa 2: Configurar a montagem
A montagem define a posicao tridimensional dos eletrodos. Vinculamos o padrao 10-20; ``on_missing="ignore"`` mantem canais adicionais (como EOG e referencias) inalterados.



In [ ]:
# Aplica as posicoes anatomicas padrao do sistema 10-20
raw.set_montage("standard_1020", on_missing="ignore")
montage = raw.get_montage()
n_pos = len(montage.ch_names) if montage is not None else 0
print(f"montage attached: standard_1020 ({n_pos} channel positions)")

## Etapa 3: Aplicar referencia media comum
Cada registro de EEG e uma diferenca de potencial eletrico. A referencia media comum subtrai a media instantanea entre todos os canais a cada ponto temporal, padrao recomendado para montagens que cobrem todo o escalpo (Cisotto & Chicco 2024 Dica 5). O parametro ``projection=False`` aplica a alteracao diretamente aos dados.



In [ ]:
# Aplica referencia media comum in-place
raw.set_eeg_reference("average", projection=False)
print(f"custom_ref_applied={raw.info['custom_ref_applied']}")

## Etapa 4: Filtro passa-banda (1-40 Hz, FIR, fase zero)
**Execute.** Um filtro FIR nao causal com janela de Hamming (metodo ``firwin``) e o padrao reprodutivel do MNE. A faixa de passagem de 1 a 40 Hz elimina oscilacoes e derivas lentas abaixo de 1 Hz e atenua ruidos de rede acima de 40 Hz.

O corte passa-alta em 1 Hz apresenta respaldo metodologico claro: tanto Kessler et al. (2025) quanto Delorme (2023) apontam que elevar o passa-alta e a escolha isolada mais eficaz para otimizar decodificadores subsequentes.



In [ ]:
# Executa a filtragem passa-banda de 1 a 40 Hz com filtro FIR de fase zero
raw.filter(
    l_freq=L_FREQ,
    h_freq=H_FREQ,
    method="fir",
    fir_design="firwin",
    phase="zero",
    verbose=False,
)
print(f"highpass={raw.info['highpass']:.2f} Hz, lowpass={raw.info['lowpass']:.2f} Hz")

**Investigue.** Plote a PSD: as derivas abaixo de 1 Hz e os componentes de alta frequencia foram atenuados, preservando o pico de ritmo alfa proximo a 10 Hz.



In [ ]:
# Calcula a densidade espectral de potencia (PSD) de Welch pos-filtragem
psd = raw.copy().pick("eeg").compute_psd(fmax=80.0, verbose=False)
fig_psd = psd.plot(picks="eeg", average=True, show=False)
# Aplica tema visual padronizado com metadados do dataset
style_figure(
    fig_psd,
    title="PSD after 1-40 Hz band-pass",
    subtitle=(
        f"{DATASET} sub-{SUBJECT} | {len(raw.copy().pick('eeg').ch_names)} EEG channels"
    ),
    source=(f"EEGDash plot_10 | OpenNeuro {DATASET} (doi:10.18112/openneuro.ds002718)"),
)
plt.show()

## Etapa 5: Reamostrar para 200 Hz
A taxa original de amostragem supera o necessario para frequencias entre 1 e 40 Hz. Reamostrar a 200 Hz preserva margem confortavel acima do limite de Nyquist (100 Hz), reduz o consumo de memoria proporcionalmente e alinha os dados as frequencias calibradas do ASR (100, 128, 200, 250, 256, 300, 500, 512 Hz).



In [ ]:
# Armazena a taxa de amostragem anterior para comparacao
sfreq_before = raw.info["sfreq"]
# Aplica a reamostragem do sinal para 200 Hz
raw.resample(TARGET_SFREQ, verbose=False)
print(f"sfreq: {sfreq_before:.1f} Hz -> {raw.info['sfreq']:.1f} Hz")
print(f"raw.get_data().shape -> {raw.get_data().shape}")

## Etapa 6: Aplicar a mesma receita ao conjunto de dados (e criar janelas)
**Execute.** As quatro etapas com :class:`~braindecode.preprocessing.Preprocessor` sao aplicadas no objeto de dataset, preservando os metadados associados. Em seguida, :func:`~braindecode.preprocessing.create_fixed_length_windows` gera janelas de 400 amostras (``WINDOW_SIZE_S * TARGET_SFREQ``) com passo identico ao tamanho da janela (0% de sobreposicao).



In [ ]:
# Calcula o numero de amostras por janela (2.0s * 200 Hz = 400 amostras)
WINDOW_SAMPLES = int(WINDOW_SIZE_S * TARGET_SFREQ)
# Executa o pipeline completo atraves do wrapper preprocess do Braindecode
preprocess(
    dataset,
    [
        Preprocessor("set_montage", montage="standard_1020", on_missing="ignore"),
        Preprocessor("set_eeg_reference", ref_channels="average"),
        Preprocessor(
            "filter",
            l_freq=L_FREQ,
            h_freq=H_FREQ,
            method="fir",
            fir_design="firwin",
        ),
        Preprocessor("resample", sfreq=TARGET_SFREQ),
    ],
)
# Fatiar o sinal continuo em janelas de tamanho fixo
windows = create_fixed_length_windows(
    dataset,
    window_size_samples=WINDOW_SAMPLES,
    window_stride_samples=WINDOW_SAMPLES,
    drop_last_window=True,
)
# Inspeciona o formato da primeira janela resultante
x0, _, _ = windows[0]
pd.DataFrame(
    {
        "value": [
            len(windows),
            f"{x0.shape}",
            str(x0.dtype),
            int(TARGET_SFREQ),
            WINDOW_SAMPLES,
        ]
    },
    index=["n_windows", "windows[0][0].shape", "dtype", "sfreq (Hz)", "window_samples"],
)

## Etapa 7: Limpeza automatizada em chamada unica com :class:`~braindecode.preprocessing.EEGPrep`
O Braindecode disponibiliza a classe :class:`~braindecode.preprocessing.EEGPrep`, que encapsula o pipeline ``clean_rawdata`` do EEGLAB. Ela encadeia remocao de offset DC, reamostragem opcional, rejeicao de canais planos, filtro passa-alta com banda de transicao configuravel, deteccao de canais ruins por correlacao espacial, reconstrucao de transientes por ASR (Mullen et al. 2015; Kothe & Makeig 2013), rejeicao de janelas com artefatos excessivos e referencia media comum.



In [ ]:
# Recarrega uma gravacao limpa e limita a 30s para diagnostico rapido em tempo de execucao
record_pp = EEGDashDataset(
    cache_dir=CACHE_DIR, dataset=DATASET, subject=SUBJECT, task=TASK
).datasets[0]
raw_full = record_pp.raw.load_data().copy()
raw_full.set_montage("standard_1020", on_missing="ignore")
raw_pp = raw_full.copy().crop(0.0, min(EEGPREP_SLICE_S, raw_full.times[-1]))
raw_before = raw_pp.copy().pick("eeg")

# Configura e executa o pipeline unificado EEGPrep
eegprep = EEGPrep(
    resample_to=TARGET_SFREQ,
    highpass_frequencies=(0.25, 0.75),
    bad_channel_corr_threshold=0.8,
    burst_removal_cutoff=10.0,
    bad_window_max_bad_channels=0.25,
    bad_channel_reinterpolate=False,
    common_avg_ref=True,
)
eegprep.fn(raw_pp)  # Executa modificacoes in-place no objeto raw_pp
raw_after = raw_pp

# Quantifica canais rejeitados e anotacoes de janelas ruins identificadas
n_dropped = len(raw_before.ch_names) - len(raw_after.copy().pick("eeg").ch_names)
n_bad_annot = sum(1 for a in raw_after.annotations if "BAD" in a["description"].upper())
pd.DataFrame(
    {
        "value": [
            len(raw_before.ch_names),
            len(raw_after.copy().pick("eeg").ch_names),
            n_dropped,
            n_bad_annot,
            f"{raw_after.info['sfreq']:.1f}",
        ]
    },
    index=[
        "n_channels (before)",
        "n_channels (after)",
        "n_channels dropped",
        "n bad-window annotations",
        "sfreq (Hz, after)",
    ],
)

## Etapa 7b: Painel diagnostico de 4 graficos antes/depois
A figura compara um trecho de 30 s da gravacao antes do EEGPrep (canto superior esquerdo) com o mesmo trecho apos o processamento (canto superior direito) em escalas identicas, a sobreposicao das curvas de PSD (canto inferior esquerdo) e o diagrama de estagios aplicados (canto inferior direito).



In [ ]:
# Importa a rotina de plotagem diagnostica auxiliar
from _eegprep_diagnostic import draw_eegprep_diagnostic  # noqa: E402

# Renderiza o painel visual comparativo antes e depois do EEGPrep
fig_diag = draw_eegprep_diagnostic(
    raw_before=raw_before,
    raw_after=raw_after,
    sfreq=raw_full.info["sfreq"],
    subject=SUBJECT,
    dataset=DATASET,
    plot_id="plot_10",
    slice_seconds=EEGPREP_SLICE_S,
    slice_start=0.0,
)
plt.show()

## Um erro comum e como se recuperar
**Execute.** Requisitar um corte passa-baixa acima da frequencia de Nyquist e um erro frequente ao reaproveitar pipelines entre gravacoes com taxas de amostragem distintas. O MNE captura essa inconsistencia com um ``ValueError``; disparamos isso deliberadamente para evidenciar a falha.



In [ ]:
# Demonstracao de falha ao definir h_freq acima da frequencia de Nyquist
try:
    raw.copy().filter(l_freq=L_FREQ, h_freq=raw.info["sfreq"], verbose=False)
except (ValueError, RuntimeError) as exc:
    print(f"Caught {type(exc).__name__}: {str(exc)[:120]}")
    nyq = raw.info["sfreq"] / 2.0
    print(
        f"Recovery: keep h_freq < Nyquist ({nyq:.1f} Hz at sfreq={raw.info['sfreq']:.0f} Hz)."
    )

## Modifique
**Modifique.** Reexecute a Etapa 4 com ``L_FREQ, H_FREQ = 1.0, 8.0`` para isolar a banda delta-teta. Observe o desaparecimento do pico de alfa na PSD. Em seguida, reexecute a Etapa 7 com ``burst_removal_cutoff=20.0`` e observe a diminuicao de anotacoes de janelas ruins a medida que o criterio do ASR se torna mais conservador.



## Mini-projeto
**Mini-projeto.** Aplique as etapas acima ao ``subject="013"`` e confirme a correspondencia de formatos nas janelas geradas. Em seguida, encapsule o EEGPrep em um :class:`~braindecode.preprocessing.Preprocessor` dentro de :func:`~braindecode.preprocessing.preprocess` para processar todos os registros de uma coorte de forma uniforme.



## Conclusao e referencias
Apresentamos o pipeline de pre-processamento canonico e a geracao de janelas. O proximo tutorial (:doc:`/generated/auto_examples/tutorials/10_core_workflow/plot_11_leakage_safe_split`) trata de esquemas de divisao de dados sem vazamento de participantes.

